# Titanic With Pipeline


In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder #for Missing values
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier

In [3]:
data=pd.read_csv("train[1].csv")
df=pd.DataFrame(data)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.drop(["PassengerId","Name","Ticket","Cabin"],axis=1,inplace=True) 
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [16]:
#Splitng the data
X_train,X_test,Y_train,y_test=train_test_split(df.drop(columns=["Survived"]),df["Survived"],test_size=0.2,random_state=42)

In [17]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [22]:
Y_train.sample(5)

285    0
563    0
425    0
795    0
525    0
Name: Survived, dtype: int64

In [23]:
X_train.isnull().sum()

Pclass        0
Sex           0
Age         140
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [7]:
#imputation missing values by simpleimputer
trf1=ColumnTransformer([
    ("imputre_age",SimpleImputer(),[2]),#Its a good practice to give index of column rather then giving column name (age index is 2)
    ("impute_embarked",SimpleImputer(strategy="most_frequent"),[6])],remainder="passthrough")



In [8]:
#one hot encoding
trf2=ColumnTransformer([("one_hot_sex_embarked",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),[1,6])],remainder="passthrough")

In [9]:
#scaling
trf3=ColumnTransformer([("scale",MinMaxScaler(),slice(0,10,None))])#slice joh hota hai vo columns wise scling krta h upper se process se 5 se 10 columns ho jaingai(0 se 9 tk hoga)

In [10]:
#feature selection
trf4=SelectKBest(score_func=chi2,k=8)

In [11]:
#Model traing
trf5=DecisionTreeClassifier()

# Create Pipeline

In [12]:
pipe=Pipeline([
    ("trf1",trf1),
    ("trf2",trf2),
    ("trf3",trf3),
    ("trf4",trf4),
    ("trf5",trf5)])

In [116]:
ans=pipe.fit(X_train,Y_train)


In [117]:
pipe.named_steps["trf1"].transformers_[1][1].statistics_

array(['S'], dtype=object)

In [118]:
Y_pred=pipe.predict(X_test)

In [120]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,Y_pred)*100

62.56983240223464

# Cross validation

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe,X_train,Y_train,cv=10,scoring="accuracy").mean() #10 bar train test split karega or un sbki accuracy ka mean dega

np.float64(0.6391431924882629)